# Library Import

In [4]:
import gymnasium as gym
import torch
from torch import nn
import torch.optim as optim
from torch.distributions import Normal      # normal dist

import numpy as np
import matplotlib.pyplot as plt
from statistics import mean, stdev
import random

import re, os, json, time
from datetime import datetime
from collections import deque
from tqdm import tqdm

# --- import the custom-made TD3 algorithm
import sys
sys.path.insert(0,'..')
from algos import TD3

# Walker2D Training with TD3

In [5]:
model_registry = {
    'TD3_v0': {
        'actor_config': [256, 256],
        'critic_config': [256, 256]
    },
    'TD3_v1': {
        'actor_config': [256, 256, 256, 256],
        'critic_config': [256, 256, 256, 256]
    },
    'TD3_v2': {
        'actor_config': [400, 300],
        'critic_config': [400, 300]
    },
    'TD3_v3': {
        'actor_config': [512, 256],
        'critic_config': [512, 256]
    }
}

MODEL_NAME = 'TD3_v0'
ALPHA1 = 1e-3
ALPHA2 = 1e-3
BETA = 1e-3
GAMMA = 0.99
TAU_C = 5e-3
TAU_A = 5e-3
SIGMA = 0.1
CLIP = 0.2

BUFFER_SIZE = 200_000
BUFFER_INIT = 10_000
BATCH_SIZE = 256
  
UPDATE_FREQ = 2
UPDATE_STEP = 2
TRAIN_ITER = 1_000_000
TRAIN_CRIT = {"pass_limit": 5, "pass_score": 2_500, 'coeff_var_limit': 1.0}
RESULT_FOLDER = 'walker_TD3_results'
CUDA_ENABLED = True
EARLY_STOP = True

In [ ]:
env = gym.make("Walker2d-v5",
               forward_reward_weight=1,     # weighting factor of the moving forward reward
               ctrl_cost_weight=1e-3,        # weighting factor of the large action penalty
               healthy_reward=1,
               terminate_when_unhealthy=True,
               healthy_z_range=(0.8,2),
               healthy_angle_range=(-1,1),
               reset_noise_scale=5e-3,       # scale of random pertubations in the initial state
               exclude_current_positions_from_observation=True
               )

for i in range(1):    
    seed = np.random.randint(1,100)
    TD3_experiment = TD3(model_name = MODEL_NAME, model_registry=model_registry, env=env,
                     alpha1=ALPHA1,alpha2=ALPHA2,beta=BETA,gamma=GAMMA,
                     tau_c=TAU_C,tau_a=TAU_A,sigma=SIGMA,clip=CLIP,
                     buffer_size=BUFFER_SIZE,buffer_init=BUFFER_INIT, batch_size=BATCH_SIZE, 
                     update_f=UPDATE_FREQ, update_step=UPDATE_STEP, iter=TRAIN_ITER,
                     seed=seed,
                     train_crit=TRAIN_CRIT,
                     result_folder=RESULT_FOLDER,
                     cuda_enabled=CUDA_ENABLED)                 
    TD3_experiment.train(early_stop=EARLY_STOP,verbose=True)

run_00002:   0%|▏                                           | 4401/1000000 [00:12<46:27, 357.11it/s]